# Publication activity and AI-text classifications

Analysis of stored detector predictions for Section 3.2, Section 4.1.1, Figure 2, and the detector-output analyses of Appendix A.1. No detector calls are made. All computations are deterministic; no random seed is needed.

## Private inputs

Supply the following files under each of `data/pred/` and `data/norm/`. Data are not included. Load pickle inputs only from a trusted source. Additional columns are ignored; identifying text stays local.

| File | Required columns and types | Meaning |
|---|---|---|
| papers.csv | publication_date: nullable number/string | Date used for all-paper publication counts |
| papers.pkl | id: unique integer; publication_date: nullable number/string; title, description: nullable strings | Dataset-local identifier, publication date, title, and abstract; text is used locally for duplicate resolution |
| checkGPT.pkl, zerogpt_results.pkl, sapling_results.pkl | paper_id: integer; gpt_generated_decision: nullable number; gpt_generated_gpt_probability: nullable number | Stored decision 0=AI, 1=Human; finite GPT score in 0–100 for CheckGPT/ZeroGPT or 0–1 for Sapling |

Run all cells from the repository root or its `notebooks` directory. Outputs remain in ignored local directories.

## Configuration

In [ ]:
from pathlib import Path
import re
import unicodedata
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from scipy.stats import norm

REPO_ROOT = Path('..') if Path.cwd().name == 'notebooks' else Path('.')
DATA_DIR = REPO_ROOT / 'data'
OUTPUT_DIR = REPO_ROOT / 'outputs'
FIG_DIR = REPO_ROOT / 'figures'
OUTPUT_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)
YEARS = list(range(2018, 2025))
DETECTORS = ['CheckGPT', 'ZeroGPT', 'Sapling']
FILES = dict(zip(DETECTORS, ['checkGPT.pkl', 'zerogpt_results.pkl', 'sapling_results.pkl']))
DATASETS = {'Predatory': 'pred', 'Normal': 'norm'}
STRATEGIES = DETECTORS + ['Majority', 'Unanimous', 'Any-positive']

def read_input(relative):
    path = DATA_DIR / relative
    return pd.read_csv(path) if path.suffix == '.csv' else pd.read_pickle(path)

## 1. Publication activity — Section 3.2

Count all CSV paper records with a recoverable publication year in 2018–2024, independently of detector coverage. Extract the first standalone four-digit year from each date. Missing/unrecoverable dates and years outside the window are excluded. Publication counts do not use the AI-analysis content-duplicate restriction.

Period totals cover 2018–2022 and 2023–2024. For annual count C, growth is (C_current − C_previous) / C_previous. Mean annual growth is the arithmetic mean of these fractional changes, expressed as a percentage. The pre-period averages transitions ending in 2019–2022; the post-period averages transitions ending in 2023 and 2024, using the 2022 count as the baseline for 2023. Zero-baseline changes are undefined and omitted.

In [ ]:
publication_counts = {}
period_rows = []
for dataset, group in DATASETS.items():
    papers = read_input(Path(group) / 'papers.csv')
    year = pd.to_numeric(papers.publication_date.astype('string').str.extract(r'(?<!\d)(\d{4})(?!\d)', expand=False), errors='coerce')
    counts = year[year.between(2018, 2024)].astype(int).value_counts().reindex(YEARS, fill_value=0).sort_index()
    publication_counts[dataset] = counts
    growth = counts.pct_change(fill_method=None).replace([np.inf, -np.inf], np.nan)
    for period, count_years, growth_years in [
        ('2018-2022', list(range(2018, 2023)), list(range(2019, 2023))),
        ('2023-2024', [2023, 2024], [2023, 2024])]:
        period_rows.append(dict(dataset=dataset, period=period, paper_count=int(counts.loc[count_years].sum()), mean_annual_growth_percent=100*growth.loc[growth_years].mean()))
annual_publications = pd.DataFrame(publication_counts).rename_axis('year').reset_index()
period_summary = pd.DataFrame(period_rows)
annual_publications.to_csv(OUTPUT_DIR / 'publication_annual_counts.csv', index=False)
period_summary.to_csv(OUTPUT_DIR / 'publication_period_summary.csv', index=False)
print(annual_publications.to_string(index=False))
print(period_summary.to_string(index=False))

## Detector preparation

No explicit abstract-length cutoff is applied. Stored prediction availability determines detector coverage; the available source does not specify original minimum-length thresholds. A short abstract can be included when its detector predictions are valid.

Analysis labels are 1=AI-positive and 0=Human-labeled. Stored binary decisions are reversed to this encoding only when the accompanying score is finite and within its existing scale. Scores are not rethresholded. Agreeing repeated predictions collapse; an invalid or conflicting repeated record invalidates that detector for the paper.

Identical nonempty normalized title+abstract groups collapse to their first input record before year filtering. Normalization uses Unicode NFKC, case folding, and nonword-to-space substitution. Available agreeing group labels resolve each detector; conflicting group labels become missing. Title-only matches and assignments across sets remain. This operational rule does not establish bibliographic identity.

AI-analysis years are numerically converted after removing U+200E and truncated to integers, then restricted to 2018–2024. IDs are dataset-local. Majority requires three valid detectors and at least two positive labels. Missing predictions never become Human.

In [ ]:
def normalized(value):
    if pd.isna(value):
        return ''
    return re.sub(r'\W+', ' ', unicodedata.normalize('NFKC', str(value)).casefold()).strip()

frames = {}
for dataset, group in DATASETS.items():
    papers = read_input(Path(group) / 'papers.pkl')
    assert papers.id.notna().all() and papers.id.is_unique
    d = papers[['id', 'publication_date', 'title', 'description']].copy()
    d['dataset'] = dataset
    d['year'] = np.trunc(pd.to_numeric(d.publication_date.astype(str).str.replace('\u200e', '', regex=False).str.strip(), errors='coerce'))
    for detector, filename in FILES.items():
        x = read_input(Path(group) / filename)
        raw = pd.to_numeric(x.gpt_generated_decision, errors='coerce')
        prob = pd.to_numeric(x.gpt_generated_gpt_probability, errors='coerce')
        valid = raw.isin([0, 1]) & prob.between(0, 1 if detector == 'Sapling' else 100) & np.isfinite(prob)
        x = x.assign(label=raw.where(valid), invalid=~valid)
        grouped = x.groupby('paper_id').label
        conflict = grouped.nunique().gt(1)
        resolved = grouped.first().mask(conflict | x.groupby('paper_id').invalid.any())
        d[detector] = d.id.map(resolved).map({0: 1., 1: 0.})
    title = d.title.map(normalized)
    abstract = d.description.map(normalized)
    key = title + '|' + abstract
    repeated = key.duplicated(False) & title.ne('') & abstract.ne('')
    for _, block in d[repeated].groupby(key[repeated]):
        keeper = block.index[0]
        for detector in DETECTORS:
            values = block[detector].dropna().unique()
            d.loc[keeper, detector] = values[0] if len(values) == 1 else np.nan
    dropped = repeated & key.duplicated()
    d = d[~dropped].copy()
    d['complete'] = d[DETECTORS].notna().all(axis=1)
    total = d[DETECTORS].sum(axis=1)
    for strategy, value in [('Majority', total.ge(2)), ('Unanimous', total.eq(3)), ('Any-positive', total.ge(1))]:
        d[strategy] = value.astype(float).where(d.complete)
    d['two_eligible'] = d[['CheckGPT', 'ZeroGPT']].notna().all(axis=1)
    d['AND'] = (d.CheckGPT.eq(1) & d.ZeroGPT.eq(1)).astype(float).where(d.two_eligible)
    d = d[d.year.between(2018, 2024)].copy()
    d['year'] = d.year.astype(int)
    frames[dataset] = d
all_labels = pd.concat(frames.values(), ignore_index=True)
assert not all_labels.duplicated(['dataset', 'id']).any()
primary = {dataset: d[d.complete].copy() for dataset, d in frames.items()}
print(pd.DataFrame([dict(dataset=dataset, n=len(d)) for dataset, d in primary.items()]).to_string(index=False))

## 2. Figure 2 — annual majority classifications

A=Predatory; B=Normal. Bars total the all-three-valid abstracts after duplicate resolution, and the ratio line uses the same yearly denominator. Human Papers and AI Papers are detector labels, not verified authorship. Empty-year ratios are undefined.

In [ ]:
annual_rows = []
for dataset, d in primary.items():
    annual = d.groupby('year').agg(n=('id', 'size'), positive=('Majority', 'sum')).reindex(YEARS, fill_value=0)
    annual['human_labeled'] = annual.n - annual.positive
    annual['ratio'] = annual.positive.div(annual.n.replace(0, np.nan))
    annual['dataset'] = dataset
    assert annual.n.sum() == len(d) and annual.positive.sum() == d.Majority.sum()
    annual_rows.append(annual.reset_index())
annual_ai = pd.concat(annual_rows, ignore_index=True)
annual_ai.to_csv(OUTPUT_DIR / 'ai_annual_counts.csv', index=False)
plt.rcParams.update({'font.size': 12, 'pdf.fonttype': 42, 'font.family': 'DejaVu Sans'})
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
max_count = annual_ai.n.max()
for ax, (panel, dataset) in zip(axes, [('A', 'Predatory'), ('B', 'Normal')]):
    a = annual_ai[annual_ai.dataset.eq(dataset)]
    ax.bar(a.year, a.human_labeled, color='#3e99bb', edgecolor='black', linewidth=.6, hatch='\\', label='Human Papers')
    ax.bar(a.year, a.positive, bottom=a.human_labeled, color='#a3245d', edgecolor='black', linewidth=.6, label='AI Papers')
    ax.set(title=f'{panel}: {dataset}', xlabel='Publication year', xticks=YEARS, ylim=(0, 1.1*max_count))
    twin = ax.twinx()
    twin.plot(a.year, a.ratio, color='#ea5566', marker='o', label='AI Ratio')
    twin.set_ylim(0, 1)
    twin.yaxis.set_major_formatter(PercentFormatter(1))
    if panel == 'A':
        twin.set_yticklabels([])
        handles, labels = ax.get_legend_handles_labels()
        line_handles, line_labels = twin.get_legend_handles_labels()
    else:
        twin.set_ylabel('AI Ratio (%)')
axes[0].set_ylabel('Paper Count')
fig.legend(handles + line_handles, labels + line_labels, loc='upper center', ncol=3, frameon=False)
fig.tight_layout(rect=[0, 0, 1, .90])
fig.savefig(FIG_DIR / 'fig2_ai_ratio.pdf', metadata={'Creator': 'Analysis notebook', 'CreationDate': None, 'ModDate': None})
plt.show()
plt.close(fig)
print(annual_ai.to_string(index=False))

## 3. AI-text prevalence — Section 4.1.1

Use the all-three-valid population. The overall comparison uses a two-sided pooled proportion z-test and signed Cohen’s h (Predatory minus Normal). Log10(p) preserves values that underflow in ordinary floating-point arithmetic. Annual rates use each dataset-year's complete-case denominator. Inference assumes independent papers despite shared profiles and detector biases; results are descriptive and noncausal.

In [ ]:
def compare(a, b, strategy):
    n, m = len(a), len(b)
    k, j = int(a[strategy].sum()), int(b[strategy].sum())
    p, q = k/n, j/m
    pooled = (k+j)/(n+m)
    z = (p-q)/np.sqrt(pooled*(1-pooled)*(1/n+1/m))
    return dict(strategy=strategy, pred_n=n, pred_positive=k, pred_percent=100*p,
                normal_n=m, normal_positive=j, normal_percent=100*q, difference_pp=100*(p-q),
                z=z, p_value=2*norm.sf(abs(z)), log10_p_value=(np.log(2)+norm.logsf(abs(z)))/np.log(10),
                cohen_h=2*(np.arcsin(np.sqrt(p))-np.arcsin(np.sqrt(q))))

majority = compare(primary['Predatory'], primary['Normal'], 'Majority')
pd.DataFrame([majority]).to_csv(OUTPUT_DIR / 'overall_majority.csv', index=False)
print(pd.DataFrame([majority]).to_string(index=False))
print(annual_ai[['dataset', 'year', 'n', 'positive', 'ratio']].to_string(index=False))

## 4. Historical detector-positive rates — 2018–2021

Compute majority-vote and Sapling positive rates separately for each dataset on all-three-valid abstracts dated 2018–2021. Historical papers are approximate negative controls, not verified human-written ground truth; these positive rates are not formal false-positive rates.

In [ ]:
historical_rows = []
for dataset, d in primary.items():
    historical_block = d[d.year.between(2018, 2021)]
    for strategy in ['Majority', 'Sapling']:
        historical_rows.append(dict(dataset=dataset, period='2018-2021', strategy=strategy, n=len(historical_block), positive=int(historical_block[strategy].sum()), percent=100*historical_block[strategy].mean()))
historical = pd.DataFrame(historical_rows)
historical.to_csv(OUTPUT_DIR / 'historical_rates.csv', index=False)
print(historical.to_string(index=False))

## 5. Detector strategies and pairwise agreement — Appendix A.1

The six strategies are CheckGPT, ZeroGPT, Sapling, Majority (at least two positive), Unanimous (three positive), and Any-positive (at least one positive). Each uses the same all-three-valid population. Pairwise agreements use that population too; the combined percentage pools abstracts across sets rather than averaging the two percentages.

Also compute the fraction of Predatory majority-positive abstracts that are positive under all three detectors. The denominator is the Predatory majority-positive count, not all eligible abstracts.

In [ ]:
strategy_rows = []
for strategy in STRATEGIES:
    row = {'strategy': strategy}
    for dataset, prefix in [('Predatory', 'pred'), ('Normal', 'normal')]:
        d = primary[dataset]
        row[prefix + '_n'] = len(d)
        row[prefix + '_positive'] = int(d[strategy].sum())
        row[prefix + '_percent'] = 100*d[strategy].mean()
    row['difference_pp'] = row['pred_percent'] - row['normal_percent']
    strategy_rows.append(row)
strategies = pd.DataFrame(strategy_rows)
strategies.to_csv(OUTPUT_DIR / 'six_strategy_comparison.csv', index=False)
pair_rows = []
for dataset, d in {**primary, 'Combined': pd.concat(primary.values(), ignore_index=True)}.items():
    for left, right in itertools.combinations(DETECTORS, 2):
        pair_rows.append(dict(dataset=dataset, detector_1=left, detector_2=right, n=len(d), agreeing=int(d[left].eq(d[right]).sum()), agreement_percent=100*d[left].eq(d[right]).mean()))
pairwise = pd.DataFrame(pair_rows)
pairwise.to_csv(OUTPUT_DIR / 'pairwise_agreement.csv', index=False)
print(strategies.to_string(index=False))
print('Difference range (percentage points):', strategies.difference_pp.min(), strategies.difference_pp.max())
print(pairwise.to_string(index=False))

predatory_positive = primary['Predatory'].loc[primary['Predatory'].Majority.eq(1)]
unanimous_count = int(predatory_positive[DETECTORS].eq(1).all(axis=1).sum())
unanimous_among_majority = pd.DataFrame([dict(dataset='Predatory', majority_positive_n=len(predatory_positive), unanimous_positive_n=unanimous_count, unanimous_percent=100*unanimous_count/len(predatory_positive))])
unanimous_among_majority.to_csv(OUTPUT_DIR / 'unanimous_among_majority.csv', index=False)
print(unanimous_among_majority.to_string(index=False))

## 6. CheckGPT AND ZeroGPT rule — Appendix A.1

Require valid predictions from CheckGPT and ZeroGPT, regardless of Sapling. The rule is positive only when both labels are AI-positive. Use the same year, prediction-resolution and content-duplicate rules. Report each dataset's positive count, eligible denominator and percentage. An AND-negative result need not mean that both detectors label the abstract Human.

In [ ]:
and_rows = []
for dataset, d in frames.items():
    eligible = d[d.two_eligible]
    and_rows.append(dict(dataset=dataset, n=len(eligible), positive=int(eligible.AND.sum()), percent=100*eligible.AND.mean()))
and_results = pd.DataFrame(and_rows)
and_results.to_csv(OUTPUT_DIR / 'two_detector_and.csv', index=False)
print(and_results.to_string(index=False))

## 7. Consensus export

Export all retained content-resolved papers dated 2018–2024, including incomplete cases, to `outputs/ai_labels.csv`. Columns are paper_id, dataset, year, CheckGPT, ZeroGPT, Sapling, majority_label. Labels are nullable integers: 1=AI-positive, 0=Human-labeled, blank=missing/unresolved. Majority is blank unless all three detectors are valid. Join using dataset and paper_id together. No identifying text is exported; citation preference and collaboration analyses are outside this notebook.

In [ ]:
export = all_labels.rename(columns={'id': 'paper_id', 'Majority': 'majority_label'})[['paper_id', 'dataset', 'year', *DETECTORS, 'majority_label']].copy()
for column in [*DETECTORS, 'majority_label']:
    export[column] = export[column].astype('Int64')
assert not export.duplicated(['dataset', 'paper_id']).any()
assert export.loc[export[DETECTORS].isna().any(axis=1), 'majority_label'].isna().all()
export.to_csv(OUTPUT_DIR / 'ai_labels.csv', index=False)
print(f'Exported {len(export)} records; data remain local.')